In [1]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score
from tqdm import tqdm

c:\Users\тема\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [20]:
base_dir = Path(r"C:\Users\тема\Desktop")
train_path = base_dir / "train_prepared.csv"
model_name = "cointegrated/rubert-tiny2"
output_dir = base_dir / f"saved_model_{model_name.split('/')[-1]}"
os.makedirs(output_dir, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
# Загрузка подготовленного датасета
df_train = pd.read_csv(train_path)
text_col = "text"
label_col = "label"

# Разделение на train/val: 80% / 20%
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df_train[text_col].values,
    df_train[label_col].values,
    test_size=0.20, 
    random_state=42,
    stratify=df_train[label_col]
)

In [ ]:
class EmotionDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=100):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = int(self.labels[idx])

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )
        item = {key: val.squeeze() for key, val in encoding.items()}
        item["labels"] = torch.tensor(label, dtype=torch.long)
        return item

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
train_dataset = EmotionDataset(train_texts, train_labels, tokenizer)
val_dataset = EmotionDataset(val_texts, val_labels, tokenizer)

batch_size = 32
max_len = 100
epochs = 10
lr = 1e-5 
patience = 3 

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=5
)
model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
total_steps = len(train_loader) * epochs
warmup_steps = int(total_steps * 0.1)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)

best_val_f1 = 0.0
best_model_state = None
emotion_names = ["joy", "sadness", "surprise", "fear", "anger"]

Loading weights: 100%|██████████| 55/55 [00:00<00:00, 8644.16it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the ch

In [24]:
for epoch in range(epochs):
    # Обучение
    model.train()
    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} (train)"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

    # Валидация
    model.eval()
    val_preds, val_true = [], []
    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} (val)"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits
            preds = torch.argmax(logits, dim=1)

            val_preds.extend(preds.cpu().numpy())
            val_true.extend(labels.cpu().numpy())

    val_f1 = f1_score(val_true, val_preds, average="macro")
    print(f"\nEpoch {epoch+1}: Val F1 (macro) = {val_f1:.4f}")

    report = classification_report(val_true, val_preds, target_names=emotion_names, digits=4)
    print("Classification report (validation):")
    print(report)

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_model_state = {k: v.cpu() for k, v in model.state_dict().items()}

# Сохранение лучшей модели
if best_model_state is not None:
    model.load_state_dict(best_model_state)
    model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)
    print(f"\n Модель сохранена в: {output_dir}")
else:
    print("Не удалось сохранить модель")

Epoch 1/10 (val): 100%|██████████| 141/141 [00:11<00:00, 12.16it/s]



Epoch 1: Val F1 (macro) = 0.3711
Classification report (validation):
              precision    recall  f1-score   support

         joy     0.6837    0.9499    0.7951      1977
     sadness     0.3948    0.6202    0.4825       998
    surprise     0.9778    0.0976    0.1774       451
        fear     0.9048    0.1672    0.2822       341
       anger     0.7541    0.0642    0.1183       717

    accuracy                         0.5897      4484
   macro avg     0.7430    0.3798    0.3711      4484
weighted avg     0.6770    0.5897    0.5161      4484



Epoch 2/10 (val): 100%|██████████| 141/141 [00:11<00:00, 12.20it/s]



Epoch 2: Val F1 (macro) = 0.7287
Classification report (validation):
              precision    recall  f1-score   support

         joy     0.7684    0.9363    0.8440      1977
     sadness     0.8306    0.5601    0.6691       998
    surprise     0.7574    0.6785    0.7158       451
        fear     0.7804    0.7713    0.7758       341
       anger     0.6657    0.6137    0.6386       717

    accuracy                         0.7625      4484
   macro avg     0.7605    0.7120    0.7287      4484
weighted avg     0.7656    0.7625    0.7542      4484



Epoch 3/10 (val): 100%|██████████| 141/141 [00:11<00:00, 12.30it/s]


...


Epoch 9: Val F1 (macro) = 0.7785
Classification report (validation):
              precision    recall  f1-score   support

         joy     0.8074    0.9181    0.8592      1977
     sadness     0.8007    0.6483    0.7165       998
    surprise     0.8114    0.7916    0.8013       451
        fear     0.8064    0.8182    0.8122       341
       anger     0.7445    0.6667    0.7035       717

    accuracy                         0.7975      4484
   macro avg     0.7941    0.7686    0.7785      4484
weighted avg     0.7962    0.7975    0.7931      4484



Epoch 10/10 (val): 100%|██████████| 141/141 [00:11<00:00, 12.22it/s]



Epoch 10: Val F1 (macro) = 0.7781
Classification report (validation):
              precision    recall  f1-score   support

         joy     0.8029    0.9231    0.8588      1977
     sadness     0.8068    0.6443    0.7164       998
    surprise     0.8157    0.7849    0.8000       451
        fear     0.8064    0.8182    0.8122       341
       anger     0.7492    0.6625    0.7032       717

    accuracy                         0.7975      4484
   macro avg     0.7962    0.7666    0.7781      4484
weighted avg     0.7967    0.7975    0.7928      4484



Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.60it/s]


 Модель сохранена в: C:\Users\тема\Desktop\saved_model_rubert-tiny2
